In [0]:
# Install boto3 if not already available
%pip install boto3 -q

import boto3
from botocore.client import Config

# S3 credentials and bucket
ACCESS_KEY = "***"
SECRET_KEY = "****"
SESSION_TOKEN = "***"
BUCKET = "de-workshop3-velasquez"

# Create S3 client with temporary credentials (works on serverless)
s3_client = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    aws_session_token=SESSION_TOKEN,
    config=Config(signature_version='s3v4')
)

print("✓ S3 client configured for serverless compute")
print(f"✓ Bucket: {BUCKET}")

# Test connection by listing objects
try:
    response = s3_client.list_objects_v2(Bucket=BUCKET, MaxKeys=10)
    if 'Contents' in response:
        print(f"✓ Successfully connected! Found {len(response['Contents'])} objects (showing first 10)")
        for obj in response['Contents'][:5]:
            print(f"  - {obj['Key']}")
    else:
        print("✓ Connected, but bucket is empty")
except Exception as e:
    print(f"✗ Connection test failed: {e}")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:725)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:443)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:443)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Cell 2 — Bronze layer (serverless compatible)
import json
from io import BytesIO

# List all JSON files in the raw/orders/ prefix
response = s3_client.list_objects_v2(Bucket=BUCKET, Prefix='raw/orders/')

if 'Contents' not in response:
    raise ValueError(f"No files found in s3://{BUCKET}/raw/orders/")

print(f"Found {len(response['Contents'])} files in S3")

# Read all JSON files into a list of records
records = []
for obj in response['Contents']:
    key = obj['Key']
    if not key.endswith('.json'):
        continue
    
    # Download file content
    file_obj = s3_client.get_object(Bucket=BUCKET, Key=key)
    content = file_obj['Body'].read().decode('utf-8')
    
    # Parse JSON lines (one JSON object per line)
    for line in content.strip().split('\n'):
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records):,} records from S3")

# Convert to Spark DataFrame
df_bronze = spark.createDataFrame(records)

print(f"\nSchema:")
df_bronze.printSchema()
print(f"\nRecord count: {df_bronze.count():,}")
display(df_bronze.limit(5))

Found 501 files in S3
Loaded 500 records from S3

Schema:
root
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- total: double (nullable = true)
 |-- unit_price: double (nullable = true)


Record count: 500


category,city,customer_id,event_ts,order_id,product,quantity,total,unit_price
Muebles,Manizales,C013,2026-05-04T20:35:32.564341+00:00,ORD-00001,Escritorio Pie,4,2480.0,620.0
Electronica,Bogota,C042,2026-05-04T20:35:33.000970+00:00,ORD-00002,Auriculares BT,1,120.0,120.0
Libros,Manizales,C010,2026-05-04T20:35:33.133363+00:00,ORD-00003,Python Avanzado,3,165.0,55.0
Electronica,Medellin,C050,2026-05-04T20:35:33.264410+00:00,ORD-00004,Auriculares BT,2,240.0,120.0
Ropa,Cali,C049,2026-05-04T20:35:33.396866+00:00,ORD-00005,Jean Clasico,3,255.0,85.0


In [0]:
# Cell 3 — Silver layer (serverless compatible)
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    # Drop any records with null order_id or negative totals (data quality)
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("total") > 0)
    # Remove duplicates: same order_id may appear if Lambda retried
    .dropDuplicates(["order_id"])
    # Parse timestamp
    .withColumn("event_ts",   F.to_timestamp("event_ts"))
    .withColumn("fecha",      F.to_date("event_ts"))
    .withColumn("hora",       F.hour("event_ts"))
    .withColumn("dia_semana", F.date_format("event_ts", "EEEE"))
    # Derived metrics
    .withColumn("descuento",  F.lit(0.0))   # placeholder for future enrichment
    .withColumn("total_neto", F.col("total"))
)

print(f"Silver record count: {df_silver.count():,}")
print("Silver DataFrame ready for use.")
display(df_silver.limit(5))

Silver record count: 500
Silver DataFrame ready for use.


category,city,customer_id,event_ts,order_id,product,quantity,total,unit_price,fecha,hora,dia_semana,descuento,total_neto
Muebles,Manizales,C013,2026-05-04T20:35:32.564Z,ORD-00001,Escritorio Pie,4,2480.0,620.0,2026-05-04,20,Monday,0.0,2480.0
Electronica,Bogota,C042,2026-05-04T20:35:33.000Z,ORD-00002,Auriculares BT,1,120.0,120.0,2026-05-04,20,Monday,0.0,120.0
Libros,Manizales,C010,2026-05-04T20:35:33.133Z,ORD-00003,Python Avanzado,3,165.0,55.0,2026-05-04,20,Monday,0.0,165.0
Electronica,Medellin,C050,2026-05-04T20:35:33.264Z,ORD-00004,Auriculares BT,2,240.0,120.0,2026-05-04,20,Monday,0.0,240.0
Ropa,Cali,C049,2026-05-04T20:35:33.396Z,ORD-00005,Jean Clasico,3,255.0,85.0,2026-05-04,20,Monday,0.0,255.0


In [0]:
# Cell 4 — Gold layer (serverless compatible)
# Use the cached df_silver from Cell 3
df_silver.createOrReplaceTempView("silver_orders")

# Gold 1: Revenue by category
gold_cat = spark.sql("""
    SELECT
        category,
        COUNT(*)                          AS num_orders,
        SUM(quantity)                     AS units_sold,
        ROUND(SUM(total_neto), 2)         AS revenue,
        ROUND(AVG(total_neto), 2)         AS avg_order_value
    FROM silver_orders
    GROUP BY category
    ORDER BY revenue DESC
""")
print("=== Revenue by category ===")
display(gold_cat)

# Gold 2: Hourly order volume (detect peak hours)
gold_hourly = spark.sql("""
    SELECT
        hora,
        COUNT(*) AS num_orders,
        ROUND(SUM(total_neto), 2) AS revenue
    FROM silver_orders
    GROUP BY hora
    ORDER BY hora
""")
print("\n=== Orders by hour ===")
display(gold_hourly)

# Gold 3: Top 10 customers by spend
gold_customers = spark.sql("""
    SELECT
        customer_id,
        COUNT(*)                    AS num_orders,
        ROUND(SUM(total_neto), 2)   AS total_spend,
        ROUND(AVG(total_neto), 2)   AS avg_order
    FROM silver_orders
    GROUP BY customer_id
    ORDER BY total_spend DESC
    LIMIT 10
""")
print("\n=== Top 10 customers ===")
display(gold_customers)

=== Revenue by category ===


category,num_orders,units_sold,revenue,avg_order_value
Electronica,137,396,261800.0,1910.95
Muebles,108,304,145520.0,1347.41
Ropa,95,278,15880.0,167.16
Libros,58,189,10395.0,179.22
Alimentos,102,313,6250.0,61.27



=== Orders by hour ===


hora,num_orders,revenue
20,500,439845.0



=== Top 10 customers ===


customer_id,num_orders,total_spend,avg_order
C020,10,22774.0,2277.4
C012,11,20568.0,1869.82
C041,9,19347.0,2149.67
C009,13,18349.0,1411.46
C035,9,18230.0,2025.56
C034,17,16385.0,963.82
C043,11,16075.0,1461.36
C030,11,14491.0,1317.36
C048,20,13739.0,686.95
C047,12,13718.0,1143.17


In [0]:
# Cell 5 — Write Gold back to S3 (serverless compatible)
import json

# Convert DataFrames to JSON and upload using boto3
def write_dataframe_to_s3(df, s3_key):
    """Convert Spark DataFrame to JSON and upload to S3"""
    # Collect to pandas, then convert to JSON lines
    pandas_df = df.toPandas()
    json_lines = pandas_df.to_json(orient='records', lines=True)
    
    # Upload to S3
    s3_client.put_object(
        Bucket=BUCKET,
        Key=s3_key,
        Body=json_lines.encode('utf-8'),
        ContentType='application/json'
    )
    print(f"✓ Uploaded to s3://{BUCKET}/{s3_key}")

# Write gold DataFrames
write_dataframe_to_s3(gold_cat, 'gold/revenue_by_category/data.json')
write_dataframe_to_s3(gold_customers, 'gold/top_customers/data.json')
write_dataframe_to_s3(gold_hourly, 'gold/hourly_orders/data.json')

print("\n✓ All gold layers written to S3 successfully!")

✓ Uploaded to s3://de-workshop3-velasquez/gold/revenue_by_category/data.json
✓ Uploaded to s3://de-workshop3-velasquez/gold/top_customers/data.json
✓ Uploaded to s3://de-workshop3-velasquez/gold/hourly_orders/data.json

✓ All gold layers written to S3 successfully!
